# MNAI Imagine 1 — 250M from scratch — Colab, step by step

**Do every cell in order, top to bottom. Do not skip one.**

Made by MeowSamOPDZN Studio. Owner: MeowSamOP DZN. Full form: Meow x Nythera AI.

This is a real image model, a 250M diffusion transformer, trained from random weights. First data is CIFAR-10, 10 classes, 64x64 pictures. It will make simple things, a cat, a dog, a car, not photoreal images yet.

## Step 1 — Check the GPU

Run this. You should see a Tesla T4, 15GB or 16GB. If you see no GPU, change the runtime type first (Runtime > Change runtime type > T4 GPU).

In [ ]:
!nvidia-smi

## Step 2 — Clone the repo

This clones the public main repo. It is public, so no login or token is needed.

In [ ]:
!git clone https://github.com/MeowSamOPDZN/mnai-imagine-1.git
%cd mnai-imagine-1
!ls -lh

## Step 3 — Install everything

This installs torch, torchvision, transformers and bitsandbytes. It takes a few minutes the first time.

In [ ]:
!pip install -q -r requirements.txt
!python -c "import torch; print('torch', torch.__version__, 'cuda?', torch.cuda.is_available())"

## Step 4 — Check the model shape

This proves the 250M model loads and runs. It should print 249,920,560 params (249.92M) and an output shape (1, 3, 64, 64). If this fails, do not go on to training, tell Muse what it said.

In [ ]:
!python model.py

## Step 5 — Save your work to Google Drive

**Do this before training.** Colab deletes everything when the session ends. This mounts Drive and makes training save there, so you can resume later.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/mnai-imagine-1
!ln -sfn /content/drive/MyDrive/mnai-imagine-1 /content/mnai-imagine-1/out-im1
!ls /content/drive/MyDrive/mnai-imagine-1

## Step 6 — Train the model (first run)

This is the real training. 1M images is the first useful target, about 20 CIFAR epochs. **It will NOT finish in one Colab session.** Colab stops after about 12 hours, sometimes sooner. Keep the laptop open, awake, plugged in, and the Colab tab open. Checkpoints save every 500 steps, so you can resume.

In [ ]:
!python train.py --out out-im1 --max-images 1000000 --micro-batch 16 --grad-accum 4

## Step 7 — Resume training (after a disconnect)

Run this instead of Step 6 when you come back. First list the checkpoints, find the newest ckpt number, put that number in the resume path below, then run it.

In [ ]:
!ls out-im1
!ls /content/drive/MyDrive/mnai-imagine-1

Change ckpt-500 below to your newest checkpoint, then run this cell:

In [ ]:
!python train.py --out out-im1 --resume out-im1/ckpt-500 --max-images 1000000 --micro-batch 16 --grad-accum 4

## Step 8 — Make pictures

Change ckpt-500 below to your newest checkpoint, then run it. It saves sample.png, a grid of 4 cats. Early checkpoints will look like noise or blurry blobs, that is normal, it gets better with training.

In [ ]:
!python generate.py --model out-im1/ckpt-500 --prompt "a photo of a cat" --n 4 --out sample.png
!ls -lh sample.png

## Step 9 — Try the other prompts

Same checkpoint, a different class. The ten it knows are in prompts.txt: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

In [ ]:
!python generate.py --model out-im1/ckpt-500 --prompt "a photo of a dog" --n 4 --out sample-dog.png
!python generate.py --model out-im1/ckpt-500 --prompt "a photo of a car" --n 4 --out sample-car.png
!ls -lh sample*.png

## Step 10 — Download your pictures

Run this, then download the files from the file browser on the left (folder icon) — right-click sample.png > Download. The checkpoint itself is in out-im1/ckpt-XXX/model.pt, about 0.5GB, save that to Drive if you want to keep training later.

In [ ]:
!ls -lh sample*.png
print('done — download sample.png from the file browser on the left')

## If something goes wrong

- **No GPU in Step 1:** Runtime > Change runtime type > T4 GPU, then Runtime > Restart runtime, run from Step 1 again.
- **Out of memory in Step 6:** lower micro-batch to 8 and raise grad-accum to 8, same 64 images per step.
- **Loss is NaN:** stop, tell Muse the last few lines, do not keep training on NaN.
- **Pictures are all noise:** that is normal for the first few hundred steps. Wait for ckpt-500 or later, then test again.
- **Session ended:** re-mount Drive in Step 5, then resume in Step 7, your checkpoints are in Drive.